# Nuclear Waste Temperature — Random Forest Optimisé
**CIVIL-226 — EPFL**

Stratégie : Random Forest avec features enrichies + interpolation spatiale par capteur voisin.

## 1. Imports

In [ ]:
import numpy as np




In [2]:
import pandas as pd
import matplotlib.pyplot as plt

In [3]:
from sklearn.neighbors import KNeighborsRegressor
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import train_test_split


np.random.seed(42)
print('Imports OK')

Imports OK


## 2. Chargement des données

In [4]:
sensors = pd.read_parquet('data/sensors.parquet')
train   = pd.read_parquet('data/train.parquet')
test    = pd.read_parquet('data/test.parquet')

sensors = sensors.drop_duplicates(subset='sensor', keep='first').reset_index(drop=True)
print(f'Doublons supprimés (N206, N213)')
print(f'Sensors : {sensors.shape} | Train : {train.shape} | Test : {test.shape}')

Doublons supprimés (N206, N213)
Sensors : (323, 4) | Train : (6626928, 4) | Test : (2190480, 3)


## 3. Preprocessing

In [5]:
# Nettoyage : NaN + aberrants AVANT le split
train_clean = train.dropna(subset=['temperature']).copy()
mask = (train_clean['temperature'] > -10) & (train_clean['temperature'] < 200)
train_clean = train_clean[mask].copy()
print(f'Lignes propres : {len(train_clean)}')
t_max = train_clean['time'].max()

Lignes propres : 6418812


## 4. Feature Engineering enrichi

On ajoute des **features d'interaction** motivées par la physique :
- `x_x_time` : la propagation du front thermique dépend de x × temps
- `dist_x_log` : atténuation logarithmique avec la distance
- `temp_neighbor` : **température du capteur train le plus proche** au même timestep — feature clé !

Cette dernière feature est la plus importante : elle donne au modèle une référence physique directe.

In [6]:
def add_features(df, sensors_df, t_max):
    """
    Joint les coordonnées et ajoute des features dérivées.
    """
    merged = df.merge(sensors_df, on='sensor', how='left')

    # Spatiales
    merged['dist_center']   = np.sqrt(merged['coor_x']**2 + merged['coor_y']**2)
    merged['dist_canister'] = np.sqrt((merged['coor_x'] - 0.7)**2 + (merged['coor_y'] - 1.2)**2)
    merged['is_opa']        = (merged['coor_x'] > 1.4).astype(float)

    # Temporelles
    merged['time_norm'] = merged['time'] / t_max
    merged['time_log']  = np.log1p(merged['time'])

    # Interactions physiques
    merged['power_x_time'] = merged['power'] * merged['time_norm']
    merged['dist_x_time']  = merged['dist_canister'] * merged['time_norm']
    merged['x_x_time']     = merged['coor_x'] * merged['time_norm']
    merged['dist_x_log']   = merged['dist_canister'] * merged['time_log']

    return merged

train_feat = add_features(train_clean, sensors, t_max)
test_feat  = add_features(test, sensors, t_max)
print('Features créées')

Features créées


In [ ]:
def add_neighbor_temperature(train_feat, test_feat, sensors):
    """
    Feature clé : température du capteur train le plus proche spatialement
    au même timestep.
    
    Principe : pour chaque point test (sensor_test, time_t),
    on trouve le capteur train le plus proche dans l'espace 2D,
    et on prend sa température à time_t comme feature.
    """
    # Positions des capteurs train et test
    train_sensors = sensors[sensors['sensor'].isin(train_feat['sensor'].unique())][['sensor','coor_x','coor_y']].drop_duplicates()
    test_sensors  = sensors[sensors['sensor'].isin(test_feat['sensor'].unique())][['sensor','coor_x','coor_y']].drop_duplicates()

    # Pour chaque capteur test, trouver le capteur train le plus proche
    neighbor_map = {}
    for _, row in test_sensors.iterrows():
        dists = np.sqrt((train_sensors['coor_x'] - row['coor_x'])**2 +
                        (train_sensors['coor_y'] - row['coor_y'])**2)
        nearest = train_sensors.iloc[dists.argmin()]['sensor']
        neighbor_map[row['sensor']] = nearest

    # Table pivot : température par (sensor_train, time)
    temp_pivot = train_feat[['sensor', 'time', 'temperature']].copy()

    # Mapper le capteur voisin sur le test
    test_feat = test_feat.copy()
    test_feat['neighbor_sensor'] = test_feat['sensor'].map(neighbor_map)

    # Joindre la température du voisin
    test_feat = test_feat.merge(
        temp_pivot.rename(columns={'sensor': 'neighbor_sensor', 'temperature': 'temp_neighbor'}),
        on=['neighbor_sensor', 'time'], how='left'
    )

    # Pour le train : le voisin le plus proche parmi les AUTRES capteurs train
    train_feat = train_feat.copy()
    train_neighbor_map = {}
    train_sensor_list = train_sensors['sensor'].tolist()

    for _, row in train_sensors.iterrows():
        others = train_sensors[train_sensors['sensor'] != row['sensor']]
        dists  = np.sqrt((others['coor_x'] - row['coor_x'])**2 +
                         (others['coor_y'] - row['coor_y'])**2)
        nearest = others.iloc[dists.argmin()]['sensor']
        train_neighbor_map[row['sensor']] = nearest

    train_feat['neighbor_sensor'] = train_feat['sensor'].map(train_neighbor_map)
    train_feat = train_feat.merge(
        temp_pivot.rename(columns={'sensor': 'neighbor_sensor', 'temperature': 'temp_neighbor'}),
        on=['neighbor_sensor', 'time'], how='left'
    )

    # Remplir les NaN avec la médiane
    median_temp = train_feat['temp_neighbor'].median()
    train_feat['temp_neighbor'] = train_feat['temp_neighbor'].fillna(median_temp)
    test_feat['temp_neighbor']  = test_feat['temp_neighbor'].fillna(median_temp)

    test_feat  = test_feat.drop_duplicates(subset=['sensor', 'time']).reset_index(drop=True)
    train_feat = train_feat.drop_duplicates(subset=['sensor', 'time']).reset_index(drop=True)
    return train_feat, test_feat

print('Calcul des températures voisines...')
train_feat, test_feat = add_neighbor_temperature(train_feat, test_feat, sensors)
print(f'temp_neighbor NaN train : {train_feat["temp_neighbor"].isna().sum()}')
print(f'temp_neighbor NaN test  : {test_feat["temp_neighbor"].isna().sum()}')

Calcul des températures voisines...
temp_neighbor NaN train : 0
temp_neighbor NaN test  : 0


In [8]:
TARGET = 'temperature'

FEATURES = ['coor_x', 'coor_y', 'time_norm', 'time_log', 'power',
            'dist_center', 'dist_canister', 'is_opa',
            'power_x_time', 'dist_x_time', 'x_x_time', 'dist_x_log',
            'temp_neighbor']

X = train_feat[FEATURES].values
y = train_feat[TARGET].values

print(f'X shape : {X.shape} — {len(FEATURES)} features')

X shape : (18778399, 13) — 13 features


## 5. Split & Normalisation

In [9]:
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)

scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_val_s   = scaler.transform(X_val)
X_test_s  = scaler.transform(test_feat[FEATURES].values)

assert len(X_test_s) == len(test)
print(f'Train : {X_train_s.shape} | Val : {X_val_s.shape} | Test : {X_test_s.shape}')

AssertionError: 

## 6. Random Forest optimisé

In [ ]:
np.random.seed(42)
idx_rf = np.random.choice(len(X_train_s), size=500_000, replace=False)

rf = RandomForestRegressor(
    n_estimators=200,
    max_depth=25,
    min_samples_leaf=2,
    max_features=0.7,
    n_jobs=-1,
    random_state=42
)
rf.fit(X_train_s[idx_rf], y_train[idx_rf])

y_pred_val = rf.predict(X_val_s)
rmse_rf = np.sqrt(mean_squared_error(y_val, y_pred_val))
print(f'Random Forest optimisé — RMSE validation : {rmse_rf:.4f}')

In [ ]:
# Feature importance
importances = pd.Series(rf.feature_importances_, index=FEATURES).sort_values(ascending=True)
importances.plot(kind='barh', figsize=(7, 5), color='steelblue')
plt.title('Feature Importance — Random Forest')
plt.xlabel('Importance')
plt.tight_layout()
plt.show()

## 7. Soumission

In [ ]:
y_pred = rf.predict(X_test_s)

submission = pd.DataFrame({
    'Id': np.arange(len(test), dtype=int),
    'temperature': y_pred.astype(float)
})

assert list(submission.columns) == ['Id', 'temperature']
assert len(submission) == len(test)
assert np.isfinite(submission['temperature']).all()
assert submission.isna().sum().sum() == 0

submission.to_csv('submission.csv', index=False)
print(f'Random Forest optimisé — RMSE validation : {rmse_rf:.4f}')
print(f'submission.csv sauvegardé — {len(submission)} lignes')
display(submission.head())